# NDVI · NDMI · VHI Time Series for a Point Location

Extracts NDVI, NDMI (from swissEO S2-SR v200) and VHI (from swissEO VHI v100)  
for a single pixel containing a given EPSG:2056 coordinate, for all available  
acquisitions in **2025**.

- **NDVI** = (B08 − B04) / (B08 + B04)
- **NDMI** = (B08 − B11) / (B08 + B11)
- **VHI** read directly from pre-calculated COGTIFFs (forest mask, v100)

Cloud, snow and terrain-shadow masks are applied to S2-SR pixels (same logic as  
the production VHI script). Masked or missing pixels appear as gaps in the chart.

## 0 · Imports & configuration

In [ ]:
# Standard library
import warnings
warnings.filterwarnings('ignore')
import os
from datetime import datetime, timedelta

# Geo / raster
import numpy as np
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import transform as warp_transform
from pystac_client import Client

# Plotting
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Allow unsigned S3 access (public buckets)
os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'

print('Imports OK')

## 1 · User inputs

In [ ]:
# ── Coordinate in EPSG:2056 (Swiss LV95) ──────────────────────────────────────
COORD_E = 2_600_000   # Easting  (e.g. 2600000)
COORD_N = 1_200_000   # Northing (e.g. 1200000)

# ── Year to analyse ───────────────────────────────────────────────────────────
YEAR = 2025

# ── Thresholds (must match production settings) ───────────────────────────────
THRESHOLD_ILLUMINATION = 70   # degrees; above → terrain shadow / low illumination
THRESHOLD_NDSI         = 0    # NDSI ≥ this → snow

# ── S2-SR scaling constants ───────────────────────────────────────────────────
S2_NODATA        = 0
S2_SCALE_FACTOR  = 0.0001
S2_OFFSET        = -0.1

# ── VHI no-data / missing-data values ─────────────────────────────────────────
VHI_NODATA   = 255
VHI_MISSING  = 110

# ── STAC / data endpoints ─────────────────────────────────────────────────────
STAC_BASE       = 'https://data.geo.admin.ch/api/stac/v0.9/'
S2SR_COLLECTION = 'ch.swisstopo.swisseo_s2-sr_v200'
VHI_BASE_URL    = 'https://data.geo.admin.ch/ch.swisstopo.swisseo_vhi_v100'

print(f'Input coordinate: E={COORD_E}, N={COORD_N} (EPSG:2056)')
print(f'Year: {YEAR}')

## 2 · Helper functions

In [ ]:
def read_pixel_value(url: str, coord_e: float, coord_n: float,
                     band: int = 1,
                     nodata=None,
                     scale: float = 1.0,
                     offset: float = 0.0,
                     dtype=np.float32) -> float | None:
    """
    Read a single pixel from a COG at (coord_e, coord_n) in EPSG:2056.

    Returns the scaled value, or None if the file cannot be opened,
    the coordinate is outside the raster, or the pixel equals nodata.
    """
    try:
        with rasterio.open(url) as src:
            # Re-project coordinate to the raster CRS if necessary
            if src.crs.to_epsg() != 2056:
                xs, ys = warp_transform('EPSG:2056', src.crs, [coord_e], [coord_n])
                px, py = xs[0], ys[0]
            else:
                px, py = coord_e, coord_n

            row, col = rowcol(src.transform, px, py)

            # Bounds check
            if not (0 <= row < src.height and 0 <= col < src.width):
                return None

            # Read the single pixel via a 1×1 window
            window = rasterio.windows.Window(col, row, 1, 1)
            val = src.read(band, window=window, out_dtype=dtype)[0, 0]

            # Nodata check (before scaling)
            if nodata is not None and val == nodata:
                return None

            val = float(val) * scale + offset
            return val

    except Exception:
        return None


def is_masked(cloud_val, snow_val, illumination_val,
              threshold_illumination=THRESHOLD_ILLUMINATION,
              threshold_ndsi=THRESHOLD_NDSI) -> bool:
    """
    Return True if the pixel should be masked out.
    Mirrors the apply_masks() logic from the production script.
    """
    if cloud_val is not None and cloud_val != 0:
        return True
    if snow_val is not None and snow_val != 0:
        return True
    if illumination_val is not None and illumination_val > threshold_illumination:
        return True
    return False


def build_s2sr_asset_url(item_id: str, suffix: str) -> str:
    """
    Build the COG URL for a swissEO S2-SR asset.
    Pattern: https://data.geo.admin.ch/<collection>/<item_id>/swisseo_s2-sr_v200_mosaic_<item_id>_<suffix>
    """
    base = f'https://data.geo.admin.ch/{S2SR_COLLECTION}/{item_id}'
    filename = f'swisseo_s2-sr_v200_mosaic_{item_id}_{suffix}'
    return f'{base}/{filename}'


def build_vhi_url(date_str: str) -> str:
    """
    Build the COG URL for a swissEO VHI v100 forest product.
    date_str: 'YYYY-MM-DD'
    """
    ts = f'{date_str}t235959'
    return f'{VHI_BASE_URL}/{ts}/ch.swisstopo.swisseo_vhi_v100_mosaic_{ts}_forest-10m.tif'


print('Helper functions defined.')

## 3 · Discover available S2-SR acquisitions via STAC

In [ ]:
# Connect to the public swisstopo STAC API
client = Client.open(STAC_BASE)
client.add_conforms_to('COLLECTIONS')
client.add_conforms_to('ITEM_SEARCH')

# Date range: full year
year_start = f'{YEAR}-01-01'
year_end   = f'{YEAR}-12-31'

# Convert input coordinate to WGS84 for STAC bbox filter
lon, lat = warp_transform('EPSG:2056', 'EPSG:4326', [COORD_E], [COORD_N])
lon, lat = lon[0], lat[0]
# Small bbox around the point (≈ 100 m buffer) to filter items
delta = 0.001  # degrees
point_bbox = [lon - delta, lat - delta, lon + delta, lat + delta]

print(f'Point in WGS84: lon={lon:.6f}, lat={lat:.6f}')
print(f'Searching STAC collection "{S2SR_COLLECTION}" for {YEAR}…')

# Search — filter by bbox and datetime
search = client.search(
    collections=[S2SR_COLLECTION],
    bbox=point_bbox,
    datetime=f'{year_start}/{year_end}',
    max_items=500,
)

s2_items = sorted(search.item_collection(), key=lambda i: i.id)
print(f'Found {len(s2_items)} S2-SR items covering the point in {YEAR}.')

## 4 · Extract NDVI and NDMI from S2-SR COGs

In [ ]:
ndvi_results = []  # list of (datetime, float)
ndmi_results = []  # list of (datetime, float)

for item in s2_items:
    item_id = item.id

    # ── Parse acquisition datetime from item id or properties ──────────────
    try:
        acq_dt = datetime.strptime(item_id[:10], '%Y-%m-%d')
    except ValueError:
        # Fallback: use STAC datetime property
        acq_dt = datetime.fromisoformat(str(item.datetime)[:10])

    # ── Build band URLs ─────────────────────────────────────────────────────
    url_b03 = build_s2sr_asset_url(item_id, 'b03_10m.tif')
    url_b04 = build_s2sr_asset_url(item_id, 'b04_10m.tif')
    url_b08 = build_s2sr_asset_url(item_id, 'b08_10m.tif')
    url_b11 = build_s2sr_asset_url(item_id, 'b11_20m.tif')
    url_cloud = build_s2sr_asset_url(item_id, 'cloudmask_10m.tif')
    url_terrain = build_s2sr_asset_url(item_id, 'terrainmask_10m.tif')
    url_scl   = build_s2sr_asset_url(item_id, 'scl_20m.tif')

    # ── Read mask layers (raw, no scaling) ──────────────────────────────────
    cloud_val       = read_pixel_value(url_cloud,   COORD_E, COORD_N, nodata=None, scale=1, offset=0)
    illumination_val = read_pixel_value(url_terrain, COORD_E, COORD_N, nodata=None, scale=1, offset=0)

    # Snow: NDSI-based (need B03 and B11 raw) + SCL band 11
    b03_raw = read_pixel_value(url_b03, COORD_E, COORD_N, nodata=S2_NODATA, scale=S2_SCALE_FACTOR, offset=S2_OFFSET)
    b11_raw_for_snow = read_pixel_value(url_b11, COORD_E, COORD_N, nodata=S2_NODATA, scale=S2_SCALE_FACTOR, offset=S2_OFFSET)
    scl_val = read_pixel_value(url_scl, COORD_E, COORD_N, nodata=None, scale=1, offset=0)

    snow_val = None
    if b03_raw is not None and b11_raw_for_snow is not None:
        denom = b03_raw + b11_raw_for_snow
        if denom != 0:
            ndsi = (b03_raw - b11_raw_for_snow) / denom
            snow_val = 1 if ndsi >= THRESHOLD_NDSI else 0
    # SCL class 11 = snow/ice
    if scl_val is not None and int(scl_val) == 11:
        snow_val = 1

    # ── Apply masks ─────────────────────────────────────────────────────────
    if is_masked(cloud_val, snow_val, illumination_val):
        print(f'  {item_id}: masked (cloud={cloud_val}, snow={snow_val}, illum={illumination_val}) → skip')
        continue

    # ── Read spectral bands (scaled) ─────────────────────────────────────────
    b04 = read_pixel_value(url_b04, COORD_E, COORD_N, nodata=S2_NODATA, scale=S2_SCALE_FACTOR, offset=S2_OFFSET)
    b08 = read_pixel_value(url_b08, COORD_E, COORD_N, nodata=S2_NODATA, scale=S2_SCALE_FACTOR, offset=S2_OFFSET)
    b11 = read_pixel_value(url_b11, COORD_E, COORD_N, nodata=S2_NODATA, scale=S2_SCALE_FACTOR, offset=S2_OFFSET)

    # ── NDVI ────────────────────────────────────────────────────────────────
    if b04 is not None and b08 is not None:
        denom = b08 + b04
        if denom != 0:
            ndvi_val = (b08 - b04) / denom
            ndvi_results.append((acq_dt, ndvi_val))

    # ── NDMI ────────────────────────────────────────────────────────────────
    if b08 is not None and b11 is not None:
        denom = b08 + b11
        if denom != 0:
            ndmi_val = (b08 - b11) / denom
            ndmi_results.append((acq_dt, ndmi_val))

    print(f'  {item_id}: NDVI={ndvi_val:.3f}, NDMI={ndmi_val:.3f}')

print(f'\nTotal valid NDVI observations: {len(ndvi_results)}')
print(f'Total valid NDMI observations: {len(ndmi_results)}')

## 5 · Extract VHI from pre-calculated COGTIFFs

In [ ]:
vhi_results = []  # list of (datetime, float)

# Build list of all Mondays in YEAR as candidate VHI dates
# VHI is published weekly; we probe every date in the year and skip missing files.
# To keep network calls manageable, iterate weekly.
probe_date = datetime(YEAR, 1, 1)
end_of_year = datetime(YEAR, 12, 31)

print(f'Probing VHI COGTIFFs for {YEAR} (weekly, every 7 days)…')

while probe_date <= end_of_year:
    date_str = probe_date.strftime('%Y-%m-%d')
    url = build_vhi_url(date_str)

    val = read_pixel_value(
        url, COORD_E, COORD_N,
        band=1,
        nodata=None,   # handle both nodata values manually below
        scale=1,
        offset=0,
        dtype=np.uint8
    )

    if val is not None and int(val) not in (VHI_NODATA, VHI_MISSING):
        vhi_results.append((probe_date, float(val)))
        print(f'  {date_str}: VHI={val:.0f}')

    probe_date += timedelta(days=7)

print(f'\nTotal valid VHI observations: {len(vhi_results)}')

## 6 · Plot time series

In [ ]:
# ── Unpack results ──────────────────────────────────────────────────────────
ndvi_dates, ndvi_vals = zip(*ndvi_results) if ndvi_results else ([], [])
ndmi_dates, ndmi_vals = zip(*ndmi_results) if ndmi_results else ([], [])
vhi_dates,  vhi_vals  = zip(*vhi_results)  if vhi_results  else ([], [])

# ── Figure ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(
    f'Vegetation indices – E={COORD_E}, N={COORD_N} (EPSG:2056) – {YEAR}',
    fontsize=13, fontweight='bold', y=1.01
)

# Shared x-axis limits
x_min = datetime(YEAR, 1, 1)
x_max = datetime(YEAR, 12, 31)

# ── NDVI ────────────────────────────────────────────────────────────────────
ax0 = axes[0]
ax0.plot(ndvi_dates, ndvi_vals,
         color='#2e8b57', linewidth=1.2, marker='o', markersize=4,
         label='NDVI')
ax0.axhline(0, color='grey', linewidth=0.6, linestyle='--')
ax0.set_ylabel('NDVI', fontsize=11)
ax0.set_ylim(-0.2, 1.0)
ax0.set_xlim(x_min, x_max)
ax0.legend(loc='upper left', fontsize=9)
ax0.grid(True, alpha=0.3)

# ── NDMI ────────────────────────────────────────────────────────────────────
ax1 = axes[1]
ax1.plot(ndmi_dates, ndmi_vals,
         color='#4169e1', linewidth=1.2, marker='o', markersize=4,
         label='NDMI')
ax1.axhline(0, color='grey', linewidth=0.6, linestyle='--')
ax1.set_ylabel('NDMI', fontsize=11)
ax1.set_ylim(-0.6, 0.6)
ax1.set_xlim(x_min, x_max)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# ── VHI ─────────────────────────────────────────────────────────────────────
ax2 = axes[2]

# Colour-code VHI by stress level
vhi_colors = {
    'Extreme stress':  ('#b56a29', lambda v: v < 10),
    'Severe stress':   ('#ce8540', lambda v: 10 <= v < 20),
    'Moderate stress': ('#f5cd85', lambda v: 20 <= v < 30),
    'Mild stress':     ('#fff5ba', lambda v: 30 <= v < 40),
    'Watch':           ('#cbffca', lambda v: 40 <= v < 50),
    'No stress':       ('#52bd9f', lambda v: 50 <= v < 60),
    'Good':            ('#0470b0', lambda v: v >= 60),
}
# Add coloured horizontal reference bands
band_ranges = [(0,10,'#b56a29'), (10,20,'#ce8540'), (20,30,'#f5cd85'),
               (30,40,'#fff5ba'), (40,50,'#cbffca'), (50,60,'#52bd9f'), (60,100,'#0470b0')]
for lo, hi, c in band_ranges:
    ax2.axhspan(lo, hi, alpha=0.12, color=c, linewidth=0)

ax2.plot(vhi_dates, vhi_vals,
         color='#333333', linewidth=1.2, marker='o', markersize=4,
         label='VHI (forest)')
ax2.set_ylabel('VHI', fontsize=11)
ax2.set_ylim(0, 100)
ax2.set_xlim(x_min, x_max)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)

# ── X-axis formatting ───────────────────────────────────────────────────────
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax2.xaxis.set_minor_locator(mdates.WeekdayLocator())
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0, ha='center')

fig.tight_layout()
plt.savefig('vhi_ndvi_ndmi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as vhi_ndvi_ndmi_timeseries.png')

## 7 · Summary table

In [ ]:
import pandas as pd

# Build a combined DataFrame indexed by date
df_ndvi = pd.DataFrame(ndvi_results, columns=['date', 'NDVI']).set_index('date')
df_ndmi = pd.DataFrame(ndmi_results, columns=['date', 'NDMI']).set_index('date')
df_vhi  = pd.DataFrame(vhi_results,  columns=['date', 'VHI']).set_index('date')

df = df_ndvi.join(df_ndmi, how='outer').join(df_vhi, how='outer').sort_index()
df.index = df.index.strftime('%Y-%m-%d')
df = df.round(3)

print(f'Summary for E={COORD_E}, N={COORD_N} — {YEAR}')
print(df.to_string())